# Fine-Tuning (Training) — Albertina PT-PT (100M)

Actually fine-tunes `PORTULAN/albertina-100m-portuguese-ptpt-encoder` (`src.config.MODEL_REGISTRY["albertina"]`) as a 10-class `Decisao`.

## Section 1 — Setup & Data Loading

Seeds every RNG (`src.runtime.set_all_seeds`) before touching the data or the model, per the project's reproducibility convention.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path("../..").resolve()))
from src.capture import capture_full_attention_subset, capture_logits_and_cls
from src.config import MODEL_REGISTRY
from src.data import build_datasets, load_label_map, load_splits
from src.evaluate import evaluate_predictions, full_test_report
from src.finetune import build_model_and_tokenizer, history_to_dicts, train, training_args_payload
from src.runtime import get_device, set_all_seeds

set_all_seeds()

<project>/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_cfg = MODEL_REGISTRY["albertina"]
device = get_device()

df = load_splits()
LABELS = load_label_map()["labels"]

print(f"device: {device}")
print(f"split sizes: {df['split'].value_counts().to_dict()}")
print(f"labels ({len(LABELS)}): {LABELS}")

device: mps
split sizes: {'train': 7614, 'test': 1523, 'val': 1016}
labels (10): ['APELAÇÃO IMPROCEDENTE', 'CONFIRMADA', 'CONFIRMADA A DECISÃO RECORRIDA', 'CONFIRMAÇÃO', 'IMPROCEDENTE', 'NÃO PROVIDO', 'PARCIALMENTE PROCEDENTE', 'PROCEDENTE', 'REVOGADA', 'REVOGADA EM PARTE']


## Section 2 — Step 1: Initialize the Model

`src.finetune.build_model_and_tokenizer` loads Albertina via `AutoModelForSequenceClassification` with `num_labels=10`.

In [3]:
model, tokenizer = build_model_and_tokenizer(model_cfg, num_labels=len(LABELS), device=device)
model.config

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 1719.39it/s]
[transformers] DebertaForSequenceClassification LOAD REPORT from: PORTULAN/albertina-100m-portuguese-ptpt-encoder
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
classifier.bias                            | MISSING    | 
pooler.dense.weight                        | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expec

DebertaConfig {
  "architectures": [
    "DebertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6,
    "LABEL_7": 7,
    "LABEL_8": 8,
    "LABEL_9": 9
  },
  "layer_norm_eps": 1e-07,
  "legacy": true,
  "max_position_embeddings": 512,
  "max_relative_positions": -1,
  "model_type": "deberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_dropout": 0,
  "pooler_hidden_act": "gelu",
 

## Section 3 — Step 2: Configure the Training Engine

`src.data.build_datasets` tokenizes `Sumario` and attaches the numeric `label_id` for every row.

In [4]:
datasets = build_datasets(df, tokenizer, max_length=model_cfg.max_length)
{split: len(ds) for split, ds in datasets.items()}

{'train': 7614, 'val': 1016, 'test': 1523}

## Section 4 — Step 3: Set the Hyperparameters

All sourced from `model_cfg` (`src.config.MODEL_REGISTRY["albertina"]`).

In [5]:
print(f"num_train_epochs: {model_cfg.num_train_epochs}")
print(f"learning_rate: {model_cfg.learning_rate}")
print(f"weight_decay: {model_cfg.weight_decay}")
print(f"warmup_ratio: {model_cfg.warmup_ratio}")
print(f"batch_size ({device.type}): {model_cfg.batch_size(device.type)}")
print(f"gradient_accumulation_steps: {model_cfg.gradient_accumulation_steps}")

num_train_epochs: 5
learning_rate: 2e-05
weight_decay: 0.01
warmup_ratio: 0.06
batch_size (mps): 8
gradient_accumulation_steps: 4


## Section 5 — Step 4: Loss Calculation and Execution

`src.finetune.train` runs the AdamW + linear-schedule loop for 5 epochs; the model natively computes cross-entropy loss.

In [ ]:
results_dir = model_cfg.results_dir("finetune")
checkpoint_dir = results_dir / "checkpoint"

history = train(
    model,
    train_dataset=datasets["train"],
    val_dataset=datasets["val"],
    model_cfg=model_cfg,
    device=device,
    labels=LABELS,
    checkpoint_dir=checkpoint_dir,
)

In [ ]:
model.save_pretrained(checkpoint_dir)
tokenizer.save_pretrained(checkpoint_dir)

In [ ]:
batch_size = model_cfg.batch_size(device.type)

full_split_artifacts = capture_logits_and_cls(model, datasets["test"], device, batch_size=batch_size)
for name, array in full_split_artifacts.items():
    np.save(results_dir / f"{name}.npy", array)
    print(f"{name}.npy: {array.shape}")

In [ ]:
N_ATTENTION_SUBSET = 50

attention_artifacts = capture_full_attention_subset(
    model, datasets["test"], device, n_samples=N_ATTENTION_SUBSET
)
for name, array in attention_artifacts.items():
    np.save(results_dir / f"{name}.npy", array)
    print(f"{name}.npy: {array.shape}")

test_df = df[df["split"] == "test"].reset_index(drop=True)
attention_subset_ids = test_df["ImportJurisprudenciaId"].iloc[:N_ATTENTION_SUBSET].tolist()
with open(results_dir / "attention_subset_ids.json", "w", encoding="utf-8") as f:
    json.dump(attention_subset_ids, f, indent=2)

In [ ]:
test_label_ids = test_df["label_id"].tolist()
test_pred_ids = full_split_artifacts["logits"].argmax(axis=-1).tolist()
test_metrics = evaluate_predictions(test_label_ids, test_pred_ids, LABELS)
test_report = full_test_report(test_label_ids, test_pred_ids, LABELS)

metrics_payload = {
    "epoch_history": history_to_dicts(history),
    "test": test_metrics,
    "classification_report": test_report["classification_report"],
    "confusion_matrix": test_report["confusion_matrix"],
    "confusion_matrix_labels": test_report["confusion_matrix_labels"],
}
with open(results_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

test_metrics